In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l0.2 Tokens

> 🎯 **Goal:** see how text becomes numbers and back again, using the simplest possible scheme, one integer per character.

**Why this matters.** A neural network does arithmetic. It cannot add or multiply the letter `T`; it can only work with numbers. So the very first thing any language model does, before attention, before training, before anything, is convert text into a list of integers. That conversion step is called *tokenization*, and getting it right (or wrong) shapes everything downstream. In l0.1 the model already did this silently; here we do it in the open.

**The intuition.** Imagine giving every distinct character a seat number in a theater: `A` sits in seat 13, `B` in seat 14, a space in seat 1, and so on. To *encode* the word "To" you just read off the seat numbers of its letters. To *decode* a list of seat numbers, you look up who sits there and read the names back. The set of all seats is the *vocabulary*. Because our vocabulary is just the characters that appear in Shakespeare, it is small (around 65 seats), and any character not in the play has no seat at all.

**The mechanics.** A *tokenizer* holds two lookup tables. `stoi` ("string to integer") maps each character to its id; `itos` ("integer to string") maps each id back to its character. *Encoding* runs a string through `stoi` to get a list of ids; *decoding* runs ids through `itos` to rebuild the string. A *round trip* means encode then decode: for any text built only from in-vocabulary characters, the round trip returns exactly what you put in. The cell below builds the tokenizer straight from the corpus and shows one round trip.

In [ ]:
from pocketlm import CharTokenizer

corpus = open(CORPUS_PATH, encoding="utf-8").read()
tok = CharTokenizer.from_text(corpus)
print("vocab size:", tok.vocab_size)
ids = tok.encode("To be")
print("encode('To be') ->", ids)
print("decode back   ->", repr(tok.decode(ids)))

## Three pathologies of character tokenization

Character tokenization is wonderfully simple, but simplicity has a price. The next cell demonstrates three real weaknesses you should understand before trusting any tokenizer.

**The intuition.** Our theater only has seats for characters that showed up in Shakespeare. Hand it a snowman or an emoji and there is nowhere to seat them: the tokenizer falls back to a special "unknown" seat (`<unk>`), and the original character is lost forever. And because every single character costs one token, even a short English word becomes a long list of numbers, which is wasteful when you later feed these to a model that has a limited attention span.

**The mechanics.** Watch for three things in the output below: (1) an *out-of-vocabulary* (OOV) character round-trips to the wrong thing, because it was replaced by `<unk>` on the way in; (2) an emoji behaves the same way, since it is just another codepoint with no seat; (3) the word "language" needs as many tokens as it has letters. Real systems fix all three with *subword* tokenization (such as BPE, byte-pair encoding), which we build later in the course.

In [ ]:
# 1) Out-of-vocab characters map to <unk>, so the round trip is lossy.
oov = "\u2603"  # a snowman, not in Shakespeare
print("OOV round trip:", repr(tok.decode(tok.encode(oov))))
# 2) An emoji is its own codepoint here, also out of vocab.
print("emoji ids:", tok.encode("\U0001f600"))
# 3) Char level means long sequences: one token per character.
print("'language' needs", len(tok.encode("language")), "tokens")

**What just happened.** You saw the snowman and emoji come back mangled (replaced by the unknown token), and you saw "language" balloon into one token per letter. These are the exact reasons production models do not tokenize by character. *Subword* tokenization (BPE) learns common chunks like "ing" or "the" and gives them their own ids: fewer tokens per sentence, and no unknown-character holes because it can always fall back to raw bytes. We will build up to that later in the course.

⚠️ **Common pitfalls**
- Assuming encode/decode is always lossless. It is lossless only for in-vocabulary text. Feed it anything the corpus never contained and you silently lose information.
- Forgetting that token count, not character count, is what costs money and attention in real models. Shorter token lists are a feature, not a detail.

🏋️ **Try it yourself.** In the cell above, swap the snowman `"\u2603"` for a different missing character such as `"\u00e9"` (an accented e) and re-run to confirm it also round-trips wrong. Then change `"language"` to a longer word and watch the token count grow one-for-one with the letters.

For in-vocabulary text the character round trip is exact, which the assert below confirms:

In [ ]:
assert tok.decode(tok.encode("To be")) == "To be"